# common_gold
Names, business rules and helper functions for the Gold layer.
Always load it AFTER common, because it uses `full_name`, `write_log`, `EXEC_LOG`, `F` and the types from there:
```
%run ./common
%run ./common_gold
```
Sections:
1. Names
2. Business rules (tracked repositories, surge rule, event categories)
3. Helpers (overwrite a Gold table, Gold metrics log, small column helpers)

In [ ]:
from datetime import date

from pyspark.sql import Window
from pyspark.sql.types import DateType, DoubleType

## 1. Names

In [ ]:
# Gold dimensions
DIM_DATE = full_name("dim_date")
DIM_HOUR = full_name("dim_hour")
DIM_EVENT_TYPE = full_name("dim_event_type")
DIM_REPO = full_name("dim_repo")

# Gold facts
FACT_ACTIVITY_HOURLY = full_name("fact_activity_hourly")
FACT_DAILY_SUMMARY = full_name("fact_daily_summary")
FACT_REPO_DAILY = full_name("fact_repo_daily")
FACT_REPO_TREND = full_name("fact_repo_trend")

# One row per Gold table per run: rows aggregated and run time (used for FinOps)
GOLD_METRICS = full_name("gold_aggregation_metrics")

LAYER_GOLD = "Silver-to-Gold"

GOLD_TABLES = [
    DIM_DATE, DIM_HOUR, DIM_EVENT_TYPE, DIM_REPO,
    FACT_ACTIVITY_HOURLY, FACT_DAILY_SUMMARY, FACT_REPO_DAILY, FACT_REPO_TREND,
]

## 2. Business rules
Change the numbers here and nowhere else.

**Tracked repository.** Most GitHub repositories in GH Archive are personal repos with one person or one bot pushing.
"Project health" only makes sense for repos with a community, so `dim_repo`, `fact_repo_daily` and `fact_repo_trend`
keep a repo only if, over the whole loaded window, it has at least `TRACKED_MIN_ACTORS` different contributors
OR at least `TRACKED_MIN_COMMUNITY_EVENTS` community events (stars, forks, pull requests, reviews, issues, comments, releases).
`fact_activity_hourly` and `fact_daily_summary` still count ALL repositories.

**Surge.** A repo-day is a surge when it has at least `SURGE_MIN_EVENTS` events and is at least `SURGE_MIN_Z`
standard deviations above the same repo's previous 7 calendar days. If the previous 7 days were completely flat
(standard deviation 0), it must be at least `SURGE_FLAT_MULTIPLIER` times their average (a "breakout").

In [ ]:
TRACKED_MIN_ACTORS = 3
TRACKED_MIN_COMMUNITY_EVENTS = 3

COMMUNITY_EVENT_TYPES = [
    "WatchEvent", "ForkEvent", "PullRequestEvent", "PullRequestReviewEvent",
    "PullRequestReviewCommentEvent", "IssuesEvent", "IssueCommentEvent", "ReleaseEvent",
]

SURGE_MIN_EVENTS = 20
SURGE_MIN_Z = 3.0
SURGE_FLAT_MULTIPLIER = 3.0

# Event type -> business category (used by dim_event_type)
EVENT_CATEGORIES = {
    "PushEvent": "Code",
    "PullRequestEvent": "Pull requests",
    "PullRequestReviewEvent": "Pull requests",
    "PullRequestReviewCommentEvent": "Pull requests",
    "IssuesEvent": "Issues",
    "IssueCommentEvent": "Issues",
    "WatchEvent": "Popularity",
    "ForkEvent": "Popularity",
    "ReleaseEvent": "Releases",
    "CreateEvent": "Repository structure",
    "DeleteEvent": "Repository structure",
}

## 3. Helpers

In [ ]:
def write_gold_table(df, table_name, view_name):
    """
    Full refresh of one Gold table.
    INSERT OVERWRITE keeps the table definition from 00b_setup_gold (types, comments, Liquid Clustering)
    and replaces all rows in one atomic Delta commit, so a rerun always gives the same result.
    Returns the number of rows now in the table.
    """
    # Put the columns in exactly the order of the table, so INSERT ... SELECT * matches by position
    target_columns = spark.table(table_name).columns
    df.select(*target_columns).createOrReplaceTempView(view_name)
    spark.sql("INSERT OVERWRITE TABLE " + table_name + " SELECT * FROM " + view_name)
    return spark.table(table_name).count()


GOLD_METRICS_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("date_from", DateType(), True),
    StructField("date_to", DateType(), True),
    StructField("source_rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("load_timestamp", TimestampType(), True),
])


def write_gold_metrics(run_id, target_table, date_from, date_to, source_rows, rows_written, start_time):
    """Append one row to gold_aggregation_metrics: how many Silver rows were aggregated and how long it took."""
    end_time = datetime.now()
    seconds = (end_time - start_time).total_seconds()
    row = [(run_id, target_table, date_from, date_to, int(source_rows), int(rows_written),
            start_time, end_time, float(seconds), end_time)]
    spark.createDataFrame(row, GOLD_METRICS_SCHEMA).write.mode("append").saveAsTable(GOLD_METRICS)


def event_category_column():
    """Builds a CASE WHEN column from the EVENT_CATEGORIES dictionary. Unknown types become 'Other'."""
    result = F.lit("Other")
    for event_type in EVENT_CATEGORIES:
        result = F.when(F.col("event_type") == event_type, F.lit(EVENT_CATEGORIES[event_type])).otherwise(result)
    return result


def count_type(event_types, alias_name):
    """Count the rows whose event_type is in the list (used inside groupBy().agg())."""
    return F.sum(F.when(F.col("event_type").isin(event_types), 1).otherwise(0)).cast("bigint").alias(alias_name)


def count_value(column_name, value, alias_name):
    """Count the rows where a column equals a value, e.g. action = 'opened'."""
    return F.sum(F.when(F.col(column_name) == value, 1).otherwise(0)).cast("bigint").alias(alias_name)